In [2]:

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

df = pd.read_csv(url, nrows=100)

print("First 5 rows:")
print(df.head())

print("\nColumn data types:")
print(df.dtypes)

print("\nDataset shape:")
print(df.shape)


First 5 rows:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  to

In [3]:

# Step 2: Profile the entire dataset in chunks

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

chunk_size = 100_000

total_rows = 0
null_counts = None
column_types = None
column_min = {}
column_max = {}

for chunk in pd.read_csv(url, chunksize=chunk_size):

    total_rows += len(chunk)

    if column_types is None:
        column_types = chunk.dtypes.astype(str)
        null_counts = pd.Series(0, index=chunk.columns, dtype="int64")

    null_counts = null_counts.add(
        chunk.isna().sum(),
        fill_value=0
    ).astype("int64")

    for column in chunk.columns:
        values = chunk[column].dropna()

        if values.empty:
            continue

        current_min = values.min()
        current_max = values.max()

        if column not in column_min or current_min < column_min[column]:
            column_min[column] = current_min

        if column not in column_max or current_max > column_max[column]:
            column_max[column] = current_max

profile = pd.DataFrame({
    "dtype": column_types,
    "null_count": null_counts,
    "null_percentage": (null_counts / total_rows * 100).round(2),
    "min": pd.Series(column_min),
    "max": pd.Series(column_max)
})

print("Total rows:", total_rows)
print("Total columns:", len(column_types))
display(profile)


/tmp/ipykernel_1221/3703768874.py:18: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(url, chunksize=chunk_size):


Total rows: 1369765
Total columns: 18


,dtype,null_count,null_percentage,min,max
VendorID,int64,98352,7.18,1,2
tpep_pickup_datetime,str,0,0.00,2008-12-31 23:05:14,2021-02-22 16:52:16
tpep_dropoff_datetime,str,0,0.00,2008-12-31 23:07:22,2021-02-22 16:56:15
passenger_count,int64,98352,7.18,0,8
trip_distance,float64,0,0.00,0.0,263163.28
RatecodeID,int64,98352,7.18,1,99
store_and_fwd_flag,str,98352,7.18,N,Y
PULocationID,int64,0,0.00,1,265
DOLocationID,int64,0,0.00,1,265
payment_type,int64,98352,7.18,1,4


In [ ]:

# Step 3: Find data quality problems

import pandas as pd

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

chunk_size = 100_000

total_rows = 0
null_counts = None

pickup_outside_january = 0
dropoff_outside_january = 0
invalid_pickup_dates = 0
invalid_dropoff_dates = 0

negative_fares = 0
zero_fares = 0
negative_distances = 0
zero_distances = 0
zero_passengers = 0

# Track row hashes across chunks to detect duplicates
seen_hashes = set()
duplicate_rows = 0

for chunk in pd.read_csv(
    url,
    chunksize=chunk_size,
    dtype={"store_and_fwd_flag": "string"}
):
    total_rows += len(chunk)

    # Count missing values across all chunks
    if null_counts is None:
        null_counts = pd.Series(
            0, index=chunk.columns, dtype="int64"
        )

    null_counts = null_counts.add(
        chunk.isna().sum(),
        fill_value=0
    ).astype("int64")

    # Parse dates for reliable date comparisons
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"], errors="coerce"
    )
    dropoff = pd.to_datetime(
        chunk["tpep_dropoff_datetime"], errors="coerce"
    )

    invalid_pickup_dates += int(pickup.isna().sum())
    invalid_dropoff_dates += int(dropoff.isna().sum())

    pickup_outside_january += int(
        (pickup.notna() &
         ((pickup < "2021-01-01") | (pickup >= "2021-02-01"))).sum()
    )

    dropoff_outside_january += int(
        (dropoff.notna() &
         ((dropoff < "2021-01-01") | (dropoff >= "2021-02-01"))).sum()
    )

    # Fare and distance checks
    negative_fares += int((chunk["fare_amount"] < 0).sum())
    zero_fares += int((chunk["fare_amount"] == 0).sum())

    negative_distances += int((chunk["trip_distance"] < 0).sum())
    zero_distances += int((chunk["trip_distance"] == 0).sum())

    # Passenger count check
    zero_passengers += int(
        (chunk["passenger_count"] == 0).sum()
    )

    # Detect exact-row duplicate candidates across chunks
    row_hashes = pd.util.hash_pandas_object(
        chunk, index=False
    )

    for row_hash in row_hashes:
        hash_value = int(row_hash)

        if hash_value in seen_hashes:
            duplicate_rows += 1
        else:
            seen_hashes.add(hash_value)

print("Total rows checked:", total_rows)

print("\nDate checks")
print("Pickup outside January:", pickup_outside_january)
print("Dropoff outside January:", dropoff_outside_january)
print("Invalid/missing pickup dates:", invalid_pickup_dates)
print("Invalid/missing dropoff dates:", invalid_dropoff_dates)

print("\nFare and distance checks")
print("Negative fares:", negative_fares)
print("Zero fares:", zero_fares)
print("Negative distances:", negative_distances)
print("Zero distances:", zero_distances)
print("Zero passenger counts:", zero_passengers)

print("\nDuplicate check")
print("Duplicate row candidates:", duplicate_rows)

print("\nMissing values by column")
display(null_counts.to_frame(name="null_count"))


In [ ]:
import pandas as pd
from time import perf_counter

url = (
    "https://github.com/DataTalksClub/nyc-tlc-data/"
    "releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
)

start = perf_counter()

chunk = pd.read_csv(
    url,
    nrows=100_000,
    dtype={"store_and_fwd_flag": "string"}
)

print("Rows loaded:", len(chunk))
print("Columns:", len(chunk.columns))
print("Load time:", round(perf_counter() - start, 2), "seconds")

print("\nMissing values:")
print(chunk.isna().sum())

print("\nBasic checks:")
print("Negative fares:", (chunk["fare_amount"] < 0).sum())
print("Zero fares:", (chunk["fare_amount"] == 0).sum())
print("Negative distances:", (chunk["trip_distance"] < 0).sum())
print("Zero distances:", (chunk["trip_distance"] == 0).sum())
print("Zero passengers:", (chunk["passenger_count"] == 0).sum())